In [1]:
# Environment Setup
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

In [3]:
# Load and Normalize Data (CIFAR10)

# Transformations for images
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)) # Normalizes pixel values
])

# Download and load Training data
trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=64, shuffle=True)

# Download and load Test data
testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=64, shuffle=False)

In [4]:
# Define the CNN
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        # Feature extraction layers
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)

        # Classification layers
        self.fc1 = nn.Linear(32 * 8 * 8, 128)
        self.fc2 = nn.Linear(128, 10) # 10 output units for 10 classes
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = x.view(-1, 32 * 8 * 8) # Flatten the tensor
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# Instantiate model and move it to GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = SimpleCNN().to(device)

In [5]:
# Define Loss function & Optimizer
criterion = nn.CrossEntropyLoss() # Standard for multi-class classification
optimizer = optim.Adam(model.parameters(), lr=0.001) # Standard, fast-converging optimizer

In [6]:
# Model Training
epochs = 5
for epoch in range(epochs):
    running_loss = 0.0
    for i, data in enumerate(trainloader, 0):
        # FIXED: Correct way to unpack the data and move to GPU/CPU
        inputs, labels = data
        inputs, labels = inputs.to(device), labels.to(device)

        # Zero the parameter gradients
        optimizer.zero_grad()

        # Forward pass + Backward pass + Optimize
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward() # Backpropagation
        optimizer.step() # Weight update

        running_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs} - Loss: {running_loss/len(trainloader):.4f}")

print("Finished Training!")

Epoch 1/5 - Loss: 1.4176
Epoch 2/5 - Loss: 1.0852
Epoch 3/5 - Loss: 0.9430
Epoch 4/5 - Loss: 0.8445
Epoch 5/5 - Loss: 0.7630
Finished Training!


In [8]:
# Accuracy Test
correct = 0
total = 0
with torch.no_grad(): # Disable gradient calculation for testing (saves memory)
    for data in testloader:
        # FIXED: Correct way to unpack test data and move to GPU/CPU
        images, labels = data
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f'Accuracy of the network on the 10000 test images: {100 * correct / total:.2f}%')

Accuracy of the network on the 10000 test images: 69.28%


In [9]:
# Save model weights locally in the notebook dir
torch.save(model.state_dict(), 'simple_cnn_cifar10.pth')
print("Model weights successfully saved as 'simple_cnn_cifar10.pth'!")

Model weights successfully saved as 'simple_cnn_cifar10.pth'!
